# C1 — Classical ML suite

Six families (LR, decision tree, random forest, XGBoost, LightGBM, CatBoost) on the B2 evaluation set. Each was tuned by a random search of 20 configurations on a time-ordered validation fold (train < 2022, validate on 2022 with KEV as of 2023-01-01: **39 positives**), then refitted with 10 seeds and tested once. Design: Study C addendum in `reports/analysis_plan_preregistration.md`.

**This notebook reads the saved outputs.** The full run takes ~2 h: `python -m src.models.classical` (or `--tune-only`, then `--reuse-tuning`).

In [1]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
import pandas as pd
pd.set_option("display.width", 220)


## Validation search (20 configs + default per family)

In [2]:
t = pd.read_csv("outputs/c1_tuning.csv")
t.groupby("family").val_pr_auc.describe().round(4)

,count,mean,std,min,25%,50%,75%,max
family,,,,,,,,
catboost,21.0,0.0328,0.0112,0.0162,0.0248,0.0305,0.0380,0.0538
decision_tree,21.0,0.0124,0.0047,0.0024,0.0102,0.0121,0.0159,0.0216
lightgbm,21.0,0.0296,0.0115,0.0052,0.0240,0.0270,0.0345,0.0668
logreg,21.0,0.0471,0.0018,0.0438,0.0462,0.0463,0.0486,0.0501
random_forest,21.0,0.0317,0.0107,0.0178,0.0240,0.0252,0.0416,0.0505
xgboost,21.0,0.0232,0.0061,0.0080,0.0200,0.0228,0.0250,0.0391


In [3]:
pd.read_csv("outputs/c1_selected_configs.csv", index_col=0)

,config
catboost,"{""iterations"": 385, ""learning_rate"": 0.0132933..."
decision_tree,"{""max_depth"": 10, ""min_samples_leaf"": 355, ""cr..."
lightgbm,"{""n_estimators"": 730, ""learning_rate"": 0.11894..."
logreg,"{""C"": 0.0013625181657841055}"
random_forest,"{""max_depth"": 10, ""min_samples_leaf"": 31, ""max..."
xgboost,"{""n_estimators"": 529, ""learning_rate"": 0.01955..."


## Test results (10 seeds, mean ± std)

In [4]:
pd.read_csv("outputs/c1_summary.csv", index_col=[0, 1])

,,pr_auc,precision_at_10,precision_at_100,ndcg_at_100,mrr,roc_auc,brier,ece
family,variant,,,,,,,,
xgboost,tuned,0.0241 ± 0.0014,0.23 ± 0.09,0.12 ± 0.0,0.1468 ± 0.0119,0.45 ± 0.2843,0.8253 ± 0.0012,0.0013 ± 0.0,0.0005 ± 0.0
logreg,default,0.0235 ± 0.0,0.3 ± 0.0,0.13 ± 0.0,0.1495 ± 0.0,0.25 ± 0.0,0.8254 ± 0.0,0.0013 ± 0.0,0.0002 ± 0.0
catboost,tuned,0.0191 ± 0.0005,0.29 ± 0.03,0.11 ± 0.0045,0.1267 ± 0.0042,0.2117 ± 0.0269,0.83 ± 0.002,0.0014 ± 0.0,0.0014 ± 0.0
logreg,tuned,0.0188 ± 0.0,0.2 ± 0.0,0.09 ± 0.0,0.1145 ± 0.0,0.2 ± 0.0,0.8329 ± 0.0,0.0014 ± 0.0,0.0006 ± 0.0
random_forest,tuned,0.0132 ± 0.0005,0.3 ± 0.0,0.06 ± 0.01,0.077 ± 0.0084,0.2093 ± 0.0427,0.8251 ± 0.0016,0.0014 ± 0.0,0.0024 ± 0.0001
lightgbm,tuned,0.0098 ± 0.0019,0.12 ± 0.0748,0.044 ± 0.0128,0.0499 ± 0.0181,0.1872 ± 0.091,0.7731 ± 0.0124,0.0014 ± 0.0,0.0014 ± 0.0
decision_tree,tuned,0.0087 ± 0.001,0.36 ± 0.0663,0.06 ± 0.02,0.1017 ± 0.032,0.5426 ± 0.3761,0.6396 ± 0.0036,0.0014 ± 0.0,0.0019 ± 0.0001
random_forest,default,0.0082 ± 0.0005,0.19 ± 0.03,0.049 ± 0.003,0.0582 ± 0.0044,0.162 ± 0.0398,0.7377 ± 0.0111,0.0015 ± 0.0,0.0037 ± 0.0001
catboost,default,0.008 ± 0.0004,0.02 ± 0.04,0.039 ± 0.0114,0.035 ± 0.0077,0.0645 ± 0.0379,0.7951 ± 0.0037,0.0014 ± 0.0,0.0005 ± 0.0001


## H2 and H1b — paired bootstrap over test CVEs (seed-averaged scores)

In [5]:
pd.read_csv("outputs/c1_hypothesis_tests.csv", index_col=0).round(4)

,pr_auc_diff,pr_auc_ci_low,pr_auc_ci_high,pr_auc_p_not_better,p_at_100_diff,p_at_100_ci_low,p_at_100_ci_high,p_at_100_p_not_better
H2_lightgbm_vs_logreg,-0.0065,-0.0204,0.0042,0.916,-0.03,-0.08,0.02,0.890
H2_lightgbm_vs_decision_tree,0.0014,-0.0088,0.0119,0.358,-0.02,-0.07,0.04,0.808
H1b_lightgbm_vs_epss_pit,-0.0052,-0.0198,0.0064,0.850,-0.05,-0.11,0.01,0.968
secondary_xgboost_vs_logreg,0.0047,-0.0009,0.0117,0.046,0.03,0.00,0.07,0.036
secondary_xgboost_vs_decision_tree,0.0126,0.0063,0.0238,0.000,0.04,0.01,0.08,0.012
secondary_catboost_vs_logreg,0.0004,-0.0042,0.0056,0.410,0.02,0.00,0.06,0.132
secondary_catboost_vs_decision_tree,0.0083,0.0047,0.0159,0.000,0.03,0.00,0.07,0.046


## Did validation PR-AUC predict test PR-AUC? (selection reliability)

In [6]:
r = pd.read_csv("outputs/c1_results_by_seed.csv")
test = r[r.variant == "tuned"].groupby("family").pr_auc.mean().rename("test_pr_auc_tuned")
best_val = t[t.config_id != "default"].groupby("family").val_pr_auc.max().rename("val_pr_auc_selected")
default_val = t[t.config_id == "default"].set_index("family").val_pr_auc.rename("val_pr_auc_default")
default_test = r[r.variant == "default"].groupby("family").pr_auc.mean().rename("test_pr_auc_default")
pd.concat([best_val, test, default_val, default_test], axis=1).round(4).sort_values("val_pr_auc_selected", ascending=False)

,val_pr_auc_selected,test_pr_auc_tuned,val_pr_auc_default,test_pr_auc_default
family,,,,
lightgbm,0.0668,0.0098,0.0052,0.0057
catboost,0.0538,0.0191,0.0190,0.0080
random_forest,0.0505,0.0132,0.0229,0.0082
logreg,0.0501,0.0188,0.0463,0.0235
xgboost,0.0391,0.0241,0.0080,0.0076
decision_tree,0.0216,0.0087,0.0024,0.0018


## Friedman across families (46 monthly batches)

In [7]:
pd.read_csv("outputs/c1_friedman.csv", index_col=0).round(4)

,statistic,p_value,n_months,rank_xgboost,rank_catboost,rank_lightgbm,rank_logreg,rank_decision_tree,rank_random_forest
precision_at_10,13.7573,0.0172,46.0,3.0435,3.2717,3.4348,3.5109,3.7717,3.9674
ndcg_at_10,16.1720,0.0064,46.0,2.9891,3.2717,3.4565,3.4348,3.8261,4.0217
map,61.9130,0.0000,46.0,2.6739,3.0217,3.9783,2.5870,5.1522,3.5870
mrr,15.6142,0.0080,46.0,2.8913,3.3043,3.7065,3.2391,4.2935,3.5652
